In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import beta, binom, multinomial

In [ ]:
observations = 20
successes = 8

## Prior Model

Beta-Binomial

In [ ]:
a = 1
b = 1

plt.plot(np.linspace(0,1,100), beta.pdf(np.linspace(0,1,100),a=a, b=b))

Likelihood?

In [ ]:
# binom.pmf(n=20, p=?)

### Prior predictive

In [ ]:
x_sim = []
p_sim = []

for i in range(10000):
  p_new = beta.rvs(a=a,b=b,size=1)[0]
  x_new = binom.rvs(p=p_new,n=20,size=1)[0]

  p_sim.append(p_new)
  x_sim.append(x_new)


In [ ]:
plt.hist(p_sim, bins=50, density=True)
plt.plot(np.linspace(0,1,100), beta.pdf(np.linspace(0,1,100),a=a, b=b))
plt.show()

In [ ]:
pd.Series(x_sim).value_counts().sort_index().plot.bar()
plt.show()

## Studying the prior

In [ ]:
# prior over a finite set of p's

# possible p's
p = [1/8, 4/8, 7/8]

# prior over the possible p's
prior = [1/3, 1/3, 1/3]

In [ ]:
mult = multinomial.rvs(n=1, p=prior, size=1000)
mult

In [ ]:
pd.Series(np.where(mult)[1]).value_counts()

In [ ]:
sim_p = []
sim_x = []

for i in range(10000):
  prior_sim = multinomial.rvs(n=1, p=prior, size=1)[0]
  p_chosen = p[np.where(prior_sim)[0][0]]

  sim_p.append(p_chosen)

  x_sim_one = binom.rvs(n=20,p=p_chosen,size=1)[0]
  sim_x.append(x_sim_one)

In [ ]:
# possible likelihoods

for i in p:
  plt.bar(range(21), binom.pmf(range(21),p=i,n=20), label=f'p={i}')
plt.legend()
plt.show()

In [ ]:
pd.Series(sim_p).value_counts().sort_index().plot.bar()

In [ ]:
pd.Series(pd.Series(sim_x).value_counts(),index=range(21)).plot.bar()
plt.title('Marginal distribution of x (Predictive)')
plt.show()

## Back to the example

In [ ]:
a = 1
b = 1


x_sim = []
p_sim = []

for i in range(5000):
  p_new = beta.rvs(a=a,b=b,size=1)[0]
  x_new = binom.rvs(p=p_new,n=20,size=1)[0]

  p_sim.append(p_new)
  x_sim.append(x_new)

plt.figure(figsize=(15,4))
plt.subplot(1,3,1)
plt.plot(np.linspace(0,1,100), beta.pdf(np.linspace(0,1,100),a=a, b=b))
plt.title('Prior')

plt.subplot(1,3,2)
plt.hist(p_sim, bins=50)
plt.title('Prior simulation')

plt.subplot(1,3,3)
(pd.Series(x_sim).value_counts() / pd.Series(x_sim).value_counts().sum()).sort_index().plot.bar()
plt.title('Data simulation')

plt.plot(range(21), binom.pmf(range(21),n=20,p=1/2), 'r', label='Binomial p=1/2')
plt.legend()
plt.show()

## Posterior

In [ ]:
a_post = a + successes
b_post = b + observations


x_sim = []
p_sim = []

for i in range(5000):
  p_new = beta.rvs(a=a_post,b=b_post,size=1)[0]
  x_new = binom.rvs(p=p_new,n=20,size=1)[0]

  p_sim.append(p_new)
  x_sim.append(x_new)

plt.figure(figsize=(15,4))
plt.subplot(1,3,1)
plt.plot(np.linspace(0,1,100), beta.pdf(np.linspace(0,1,100),a=a_post, b=b_post))
plt.title('Posterior')

plt.subplot(1,3,2)
plt.hist(p_sim, bins=50)
plt.title('Posterior simulation')

plt.subplot(1,3,3)
pd.Series(x_sim).value_counts(normalize=True).sort_index().plot.bar()
plt.plot(range(21), binom.pmf(range(21),n=20,p=8/20), 'r', label='Binomial p=8/20')
plt.title('Posterior data simulation (Marginal of x)')
plt.legend()
plt.show()

## Bayes Factor

Hypothesis test vs p=1/2

In [ ]:
binom.pmf(8,p=1/2,n=20)

In [ ]:
binom.pmf(8,p=1/2,n=20)

In [ ]:
posterior_predictive = (pd.Series(x_sim).value_counts().sort_index() / pd.Series(x_sim).value_counts().sum())
posterior_predictive

In [ ]:
posterior_predictive[8]

In [ ]:
binom.pmf(8,p=1/2,n=20) / posterior_predictive[8]

In [ ]:
np.sqrt(10)

In [ ]:
posterior_predictive.plot.bar(label='Posterior predictive')
plt.vlines(x=8,ymin=0, ymax=binom.pmf(8,p=1/2,n=20), label='P(8|p=1/2)', color='r', linestyles='--')
plt.plot(8,binom.pmf(8,p=1/2,n=20),'o', color='red')
plt.plot(8,posterior_predictive[8],'o', color='blue')
plt.legend()
plt.show()